In [1]:
!pip install gtfparse
!pip install polars=='0.16.17'
!pip install pyarrow
!pip install anndata==0.8.0

In [ ]:
#Inputs: csv outputs of SoupX ambient RNA removal pipeline
#Ouputs: combined anndata object

In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import pyarrow
from gtfparse import read_gtf
import scipy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
#Input all csv outputs of SoupX
dat_list = ['../../SoupX/MO/Outputs_old/Run12_sample2_soupcorrected_plus5.csv',
            '../../SoupX/MO/Outputs_old/Run12_sample3_soupcorrected_plus5.csv',
           '../../SoupX/MO/Outputs_old/Run12_sample6_soupcorrected_plus5.csv',
           '../../SoupX/MO/Outputs_old/Run12_sample7_soupcorrected_plus5.csv',
            '../../SoupX/MO/Outputs_old/Run12_sample9_soupcorrected_plus5.csv',
           '../../SoupX/MO/Outputs_old/Run12_sample10_soupcorrected_plus5.csv',
           '../../SoupX/MO/Outputs_old/Run12_sample11_soupcorrected_plus5.csv',
           '../../SoupX/MO/Outputs_old/Run12_sample12_soupcorrected_plus5.csv']

In [3]:
#Input the first sample here
tot_dat = ad.read_csv('../../SoupX/MO/Outputs_old/Run12_sample1_soupcorrected_plus5.csv',)

tot_dat = tot_dat.T
tot_dat.X = scipy.sparse.csr_matrix(tot_dat.X)
counts = np.sum(tot_dat.X, axis = 1).A.reshape((1,len(tot_dat)))[0]
genes = np.count_nonzero(tot_dat.X.A, axis = 1)

tot_dat.obs['n_counts'] = counts
tot_dat.obs['n_genes'] = genes
#manually set key name
tot_dat.obs['key'] = 'Run12_sample1'
tot_dat.var_names = [i for i in tot_dat.var_names]

for dn in dat_list:
    dat = ad.read_csv(dn)
    dat = dat.T
    dat.X = scipy.sparse.csr_matrix(dat.X)
    counts = np.sum(dat.X, axis = 1).A.reshape((1,len(dat)))[0]
    genes = np.count_nonzero(dat.X.A, axis = 1)
    
    dat.obs['n_counts'] = counts
    dat.obs['n_genes'] = genes
    #set this such that it outputs the key that you would like
    dat.obs['key'] = dn.split('_')[1].split('/')[-1] + '_' + dn.split('_')[2]
    dat.var_names = [i for i in dat.var_names]
    
    tot_dat = ad.concat([tot_dat, dat], join = 'outer')
    print(tot_dat.shape)

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


(15615, 18482)
(24352, 18751)
(34979, 18963)
(43342, 19064)
(53301, 19173)
(63524, 19237)
(74818, 19280)
(84652, 19307)


In [4]:
tot_dat.var_names

Index(['ENSMOCG00000000002', 'ENSMOCG00000000003', 'ENSMOCG00000000004',
       'ENSMOCG00000000005', 'ENSMOCG00000000006', 'ENSMOCG00000000008',
       'ENSMOCG00000000009', 'ENSMOCG00000000010', 'ENSMOCG00000000011',
       'ENSMOCG00000000012',
       ...
       'ENSMOCG00000023030', 'ENSMOCG00000023031', 'ENSMOCG00000023032',
       'ENSMOCG00000023033', 'ENSMOCG00000023034', 'ENSMOCG00000023035',
       'ENSMOCG00000023036', 'ENSMOCG00000023037', 'ENSMOCG00000023038',
       'ENSMOCG00000023039'],
      dtype='object', length=19307)

In [5]:
tot_dat.obs['key'].unique()

array(['Run12_sample1', 'Run12_sample2', 'Run12_sample3', 'Run12_sample6',
       'Run12_sample7', 'Run12_sample9', 'Run12_sample10',
       'Run12_sample11', 'Run12_sample12'], dtype=object)

In [6]:
#input the gtf that you are using so that you can get gene names (only for prairie vole)
db_gene = read_gtf('../../SoupX/MO/Microtus_ochrogaster.MicOch1.0.112.gtf', features = ['gene', 'transcript'])
df_gene = db_gene.to_pandas()

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_source', 'transcript_biotype', 'tag', 'gene_name', 'transcript_name']


In [7]:
df_gene

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_version,gene_source,gene_biotype,transcript_id,transcript_version,transcript_source,transcript_biotype,tag,gene_name,transcript_name
0,1,ensembl,gene,23719295,23734994,NaN,+,0,ENSMOCG00000000281,1,ensembl,protein_coding,,,,,,,
1,1,ensembl,transcript,23719295,23734994,NaN,+,0,ENSMOCG00000000281,1,ensembl,protein_coding,ENSMOCT00000000366,1,ensembl,protein_coding,Ensembl_canonical,,
2,1,ensembl,gene,46380509,46480297,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,,,,,,Tecpr2,
3,1,ensembl,transcript,46380509,46480297,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,ENSMOCT00000000489,1,ensembl,protein_coding,Ensembl_canonical,Tecpr2,Tecpr2-201
4,1,ensembl,transcript,46387947,46478097,NaN,+,0,ENSMOCG00000000369,1,ensembl,protein_coding,ENSMOCT00000000494,1,ensembl,protein_coding,,Tecpr2,Tecpr2-202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54552,AHZW01186317.1,ensembl,transcript,594,720,NaN,-,0,ENSMOCG00000006135,1,ensembl,miRNA,ENSMOCT00000007974,1,ensembl,miRNA,Ensembl_canonical,,
54553,AHZW01186775.1,ensembl,gene,949,1039,NaN,-,0,ENSMOCG00000005747,1,ensembl,miRNA,,,,,,,
54554,AHZW01186775.1,ensembl,transcript,949,1039,NaN,-,0,ENSMOCG00000005747,1,ensembl,miRNA,ENSMOCT00000007482,1,ensembl,miRNA,Ensembl_canonical,,
54555,AHZW01186170.1,ensembl,gene,339,436,NaN,+,0,ENSMOCG00000008661,1,ensembl,miRNA,,,,,,,


In [8]:
#rename genes with gene name instead of gene ID (only for prairie vole)
fin_gene_name = {}
for item in tot_dat.var_names:
    gn = df_gene.loc[df_gene.index[df_gene['gene_id'] == item][0], 'gene_name']
    if gn != "":
        fin_gene_name[item] = gn
    else:
        fin_gene_name[item] = item

In [9]:
#check to see if gene names match names in BLAST tables
mapping = pd.read_csv('../../BLASTMAPPING/maps/active_maps/hypo_proj/mgmo/mg_to_mo.txt', delimiter = '\t', header = None)

In [10]:
tot_dat.var_names = [fin_gene_name[i] for i in tot_dat.var_names]

In [11]:
a = 0
mo_set = set(mapping[1].unique())
for item in tot_dat.var_names:
    if item in mo_set:
        a += 1
a

18026

In [12]:
tot_dat.var_names

Index(['ENSMOCG00000000002', 'ENSMOCG00000000003', 'ENSMOCG00000000004',
       'ENSMOCG00000000005', 'ND1', 'ENSMOCG00000000008', 'ENSMOCG00000000009',
       'ND2', 'ENSMOCG00000000011', 'ENSMOCG00000000012',
       ...
       'ENSMOCG00000023030', 'Gemin2', 'Creb3', 'Ppfia4', 'Vps28', 'Trp53bp2',
       'Rfc1', 'ENSMOCG00000023037', 'Hbs1l', 'Celsr1'],
      dtype='object', length=19307)

In [13]:
tot_dat.obs_names_make_unique()
tot_dat.var_names_make_unique()

In [14]:
tot_dat.obs['key'].unique()

array(['Run12_sample1', 'Run12_sample2', 'Run12_sample3', 'Run12_sample6',
       'Run12_sample7', 'Run12_sample9', 'Run12_sample10',
       'Run12_sample11', 'Run12_sample12'], dtype=object)

In [15]:
test_dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_MO_soupX_plus5.h5ad')

In [16]:
def csr_equal(A, B):
    if A.shape != B.shape:
        return False
    return (A != B).nnz == 0

In [17]:
csr_equal(test_dat.X,tot_dat.X)

True

In [16]:
#tot_dat.write('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_MO_soupX_08242026.h5ad')

... storing 'key' as categorical
